In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hamasali/grpo-stage1/README.md
/kaggle/input/datasets/hamasali/grpo-stage1/adapter_config.json
/kaggle/input/datasets/hamasali/grpo-stage1/adapter_model.safetensors
/kaggle/input/datasets/hamasali/grpo-stage1/chat_template.jinja
/kaggle/input/datasets/hamasali/grpo-stage1/tokenizer.json
/kaggle/input/datasets/hamasali/grpo-stage1/tokenizer_config.json
/kaggle/input/datasets/hamasali/grpo-stage1/training_args.bin


# HLT Project — GRPO Stage 2: 3-number puzzles (Group 2)
**Starts from:** Stage 1 LoRA (2 numbers, 100 steps) → val: 2-num 98%, 3-num 24% (CoT baseline: 28% / 4%)
**Goal:** improve 3-number accuracy with 400 GRPO steps, without forgetting 2 numbers.

| Step | What |
|---|---|
| 1 | Setup: code (branch main), libraries, data |
| 2 | Load the Stage 1 adapter (Kaggle Dataset `grpo-stage1`) |
| 3 | Train Stage 2: 3 numbers, 400 steps, checkpoint every 25 |
| 4 | Evaluate every checkpoint on val → pick the best |
| 5 | Save everything → `stage2.zip` (Output tab) |

## 1. Setup
- Clone the repo (branch `main`) and pull the newest code.
- Remove Kaggle's old `torchao` (it crashes PEFT), install TRL (GRPOTrainer) + PEFT (LoRA), print the versions.
- Check the GPU (Tesla T4, 16 GB; we use only GPU 0).
- Generate the data if missing (seed 42 → exactly the same data as the baselines and Stage 1).

In [2]:
# 1. Setup
import os, glob, json, shutil
REPO, GITHUB_USER = "hlt-grpo-countdown", "hamasli"     # <-- change this

# 1a. Code
if not os.path.exists(f"/kaggle/working/{REPO}"):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /kaggle/working/{REPO}
%cd /kaggle/working/{REPO}
!git checkout main && git pull
!ls src/train                                     # expect grpo_train.py

# 1b. Libraries
!pip uninstall -y -q torchao
!pip install -q trl peft
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__)

# 1c. GPU
print("GPU:", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# 1d. Data
if not os.path.exists("data/train.jsonl"):
    !python -m src.data.generate
!wc -l data/*.jsonl                               # train 10000, val 200, test 1000, test_ood 500

Cloning into '/kaggle/working/hlt-grpo-countdown'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (352/352), done.
remote: Compressing objects: 100% (214/214), done.
remote: Total 352 (delta 188), reused 289 (delta 126), pack-reused 0 (from 0)
Receiving objects: 100% (352/352), 4.09 MiB | 13.85 MiB/s, done.
Resolving deltas: 100% (188/188), done.
/kaggle/working/hlt-grpo-countdown
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.
grpo_train.py  __init__.py
torch 2.11.0+cu130 | transformers 5.20.2 | trl 1.18.0 | peft 0.21.2
GPU: Tesla T4 | 15.6 GB
   1000 data/test.jsonl
    500 data/test_ood.jsonl
  10000 data/train.jsonl
    200 data/val.jsonl
  11700 total


In [3]:
# 1. Setup
import os, glob, json, shutil
REPO, GITHUB_USER = "hlt-grpo-countdown", "<your-username>"     # <-- change this

# 1a. Code
if not os.path.exists(f"/kaggle/working/{REPO}"):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /kaggle/working/{REPO}
%cd /kaggle/working/{REPO}
!git checkout main && git pull
!ls src/train                                     # expect grpo_train.py

# 1b. Libraries
!pip uninstall -y -q torchao
!pip install -q trl peft
import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__)

# 1c. GPU
print("GPU:", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# 1d. Data
if not os.path.exists("data/train.jsonl"):
    !python -m src.data.generate
!wc -l data/*.jsonl                               # train 10000, val 200, test 1000, test_ood 500

/kaggle/working/hlt-grpo-countdown
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.
grpo_train.py  __init__.py
torch 2.11.0+cu130 | transformers 5.20.2 | trl 1.18.0 | peft 0.21.2
GPU: Tesla T4 | 15.6 GB
   1000 data/test.jsonl
    500 data/test_ood.jsonl
  10000 data/train.jsonl
    200 data/val.jsonl
  11700 total


## 2. Load the Stage 1 adapter
Stage 2 continues from Stage 1 (curriculum learning): the model already knows the format
and the rule "use only the given numbers". We find the adapter files inside the attached Dataset `grpo-stage1`.

In [4]:
# 2. Find the Stage 1 adapter in /kaggle/input (the attached Dataset).
hits = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
assert hits, "Stage 1 adapter not found -> Add Input: dataset 'grpo-stage1'"
STAGE1 = os.path.dirname(hits[0])
print("Stage 1 adapter:", STAGE1)
print(os.listdir(STAGE1))                         # expect adapter_config.json + adapter_model.safetensors

Stage 1 adapter: /kaggle/input/datasets/hamasali/grpo-stage1
['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [5]:
# 3a. Train Stage 2 from the Stage 1 LoRA.
#     Live: one line per 5 steps (visible in the version's Logs tab). Copy saved in logs/stage2.log.
!mkdir -p logs
!PYTHONUNBUFFERED=1 python -m src.train.grpo_train --levels 3 --max_steps 400 --init_adapter {STAGE1} --output_dir checkpoints/stage2 2>&1 | tee logs/stage2.log

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 232.50it/s]
Loading existing LoRA from /kaggle/input/datasets/hamasali/grpo-stage1
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Levels [3] | 5000 puzzles available | 8 puzzles x 4 answers = 32 answers/step | 400 steps -> 3200 puzzles used
Training: 400 steps, log every 5
step    5/400 | correct  18.8% | format  99.4% | no_signal 0.70 | length   46 tok | clipped  0.6% |   1.7 min | ETA  133 min
╭─────────────────────────────────── Step 5 ───────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt        ┃ Completion    ┃ correctness… ┃ format_rewar… ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ <|im_start|>… │ 92 - 60 = 32; │         1.00 │          0.10 │      0.00 │ │
│ │ You are Qwen, │ 32 + 1 =      │              │               │           │ │
│ │ created by    │ 33</th

In [6]:
# 4a. Evaluate every checkpoint on val (greedy) and print one line per checkpoint.
os.makedirs("results/stage2", exist_ok=True)
SUM = "results/Qwen2.5-1.5B-Instruct_grpo_val_summary.json"
ckpts = sorted(glob.glob("checkpoints/stage2/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
for c in ckpts:
    step = c.rsplit("-", 1)[1]
    !python -m src.eval.evaluate --model Qwen/Qwen2.5-1.5B-Instruct --method grpo --data data/val.jsonl --adapter {c} > /dev/null 2>&1
    shutil.copy(SUM, f"results/stage2/val_step{step}_summary.json")
    s = json.load(open(SUM))
    print(f"val step {step:>3} | 2-num {s['2_numbers']['accuracy']:5.1f}% | 3-num {s['3_numbers']['accuracy']:5.1f}%"
          f" | length {s['all']['mean_tokens']:5.1f} tok | clipped {s['all']['clipped_rate']:4.1f}%", flush=True)

val step  25 | 2-num  97.0% | 3-num  30.0% | length  44.8 tok | clipped  1.5%
val step  50 | 2-num  98.0% | 3-num  33.0% | length  41.5 tok | clipped  0.0%
val step  75 | 2-num  65.0% | 3-num  33.0% | length  38.3 tok | clipped  0.5%
val step 100 | 2-num  66.0% | 3-num  39.0% | length  35.6 tok | clipped  1.5%
val step 125 | 2-num  66.0% | 3-num  36.0% | length  33.0 tok | clipped  0.5%
val step 150 | 2-num  77.0% | 3-num  37.0% | length  31.9 tok | clipped  0.0%
val step 175 | 2-num  76.0% | 3-num  46.0% | length  35.2 tok | clipped  1.5%
val step 200 | 2-num  98.0% | 3-num  45.0% | length  41.0 tok | clipped  3.0%
val step 225 | 2-num  98.0% | 3-num  47.0% | length  39.9 tok | clipped  1.5%
val step 250 | 2-num  94.0% | 3-num  47.0% | length  38.6 tok | clipped  0.5%
val step 275 | 2-num  98.0% | 3-num  54.0% | length  44.2 tok | clipped  2.5%
val step 300 | 2-num  97.0% | 3-num  52.0% | length  39.7 tok | clipped  0.0%
val step 325 | 2-num  94.0% | 3-num  53.0% | length  39.7 tok | 

In [7]:
# !PYTHONUNBUFFERED=1 python -m src.train.grpo_train --model Qwen/Qwen2.5-0.5B-Instruct --max_steps 10 --logging_steps 5 --output_dir checkpoints/test_log 2>&1 | tee logs/test.log